# Latency Percentiles from a Capture

Companion notebook for the [Reading a Packet Capture](https://codemia.io/courses/networking_for_backend_engineers/net_reading_a_packet_capture) lesson on Codemia.

**What you will build**: a request-latency distribution with the shape a real service produces,
the percentiles that describe it, and a demonstration of why an average is the wrong number to
alert on and why a fan-out turns a good tail into a bad one.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

## Concept recap

Service latency is not symmetric. Most requests take about the same time, a few take considerably
longer, and the long ones have causes that the short ones do not: a retransmission, a garbage
collection pause, a cold cache, a queue behind a slow neighbour. A distribution like that has a
mean that sits well below the experience of a noticeable fraction of users, which is why percentiles
exist and why the interesting ones are far out.

## A synthetic capture

In [ ]:
def synthesise(n=200_000, base_ms=18, sigma=0.35, loss_rate=0.004, rto_ms=200, packets=12):
    """Lognormal service time, plus a retransmission timeout on requests that lose a packet."""
    latency = rng.lognormal(mean=np.log(base_ms), sigma=sigma, size=n)

    # a request of `packets` segments loses at least one with this probability
    hit = rng.random(n) < (1 - (1 - loss_rate) ** packets)
    latency = latency + hit * rto_ms

    # a small slice waits behind something slow: a lock, a compaction, a cold cache
    stalled = rng.random(n) < 0.0015
    latency = latency + stalled * rng.uniform(300, 1200, size=n)
    return latency, hit, stalled

latency, hit, stalled = synthesise()
print(f'{len(latency):,} requests')
print(f'{hit.mean():.2%} hit a retransmission, {stalled.mean():.3%} stalled behind something else')

In [ ]:
def percentiles(values):
    qs = [50, 75, 90, 95, 99, 99.9, 99.99]
    return {f'p{q:g}': np.percentile(values, q) for q in qs}

stats = percentiles(latency)
print(f'mean   {latency.mean():>8.1f} ms')
for name, value in stats.items():
    print(f'{name:<7}{value:>8.1f} ms')
print()
print(f'p99 is {stats["p99"] / latency.mean():.1f}x the mean')
print(f'fraction of requests slower than the mean: {(latency > latency.mean()).mean():.1%}')

The mean sits above the median and below the 90th percentile, describing nobody in particular. The
jump between p95 and p99 is the retransmission population arriving: a fixed 200ms added to a
request that would otherwise have been ordinary. That is a step in the distribution, not a slope,
and a step is the signature of a timer rather than of load.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(latency[latency < 400], bins=160)
for q, colour in [(50, 'tab:green'), (99, 'tab:orange')]:
    axes[0].axvline(np.percentile(latency, q), color=colour, linestyle='--', label=f'p{q}')
axes[0].set_xlabel('latency (ms)')
axes[0].set_ylabel('requests')
axes[0].set_title('The bulk, with the retransmission shoulder at ~220ms')
axes[0].legend()

ordered = np.sort(latency)
survival = 1 - np.arange(len(ordered)) / len(ordered)
axes[1].plot(ordered, survival)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('latency (ms)')
axes[1].set_ylabel('fraction slower')
axes[1].set_title('Tail, on log axes')
axes[1].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

The right-hand plot is the one to get used to reading. A straight line on log axes means a tail
with no characteristic scale, and the kinks are populations: one for the timer, one for whatever
stalls for half a second. Each kink has a different cause and a different fix.

## Why a fan-out ruins a good tail

In [ ]:
def fanout_latency(samples, fanout, rng):
    """A request that waits for `fanout` parallel calls takes the slowest of them."""
    picks = rng.choice(samples, size=(len(samples) // 10, fanout))
    return picks.max(axis=1)

print(f"{'fan-out':>8}{'p50':>9}{'p99':>9}{'p99.9':>9}")
for fanout in [1, 2, 5, 10, 50, 100]:
    combined = fanout_latency(latency, fanout, rng)
    print(f'{fanout:>8}{np.percentile(combined, 50):>8.1f} {np.percentile(combined, 99):>8.1f} '
          f'{np.percentile(combined, 99.9):>8.1f}')

A single backend with a 1% chance of being slow is fine. A request that fans out to a hundred of
them is almost certain to touch one, so the *median* of the fan-out approaches the *tail* of the
individual call. This is the arithmetic behind hedged requests and behind the advice to keep
fan-out narrow: the tail you have to engineer against is not the one your backends report.

## What the retransmissions alone are worth

In [ ]:
clean = latency[~hit & ~stalled]
print(f'all requests      p50 {np.percentile(latency, 50):>7.1f}   p99 {np.percentile(latency, 99):>7.1f}')
print(f'without loss      p50 {np.percentile(clean, 50):>7.1f}   p99 {np.percentile(clean, 99):>7.1f}')
print()
for loss_rate in [0.0, 0.0005, 0.002, 0.004, 0.01]:
    sample, _, _ = synthesise(n=400_000, loss_rate=loss_rate)
    print(f'loss {loss_rate:>7.2%} -> p99 {np.percentile(sample, 99):>7.1f} ms,'
          f' p99.9 {np.percentile(sample, 99.9):>7.1f} ms')

Removing the loss removes the shoulder and the 99th percentile falls back to the service's own
distribution. Notice how little loss it takes: at half a tenth of a percent the tail is already
moving, because a request is many packets and any one of them is enough. A loss rate that looks
negligible as a percentage of packets is not negligible as a percentage of requests.

## What to try next

- Replace the synthetic data with your own. Export response times from a log, load them into an
  array, and run the same percentile and fan-out analysis.
- Add a retry on the slow path and see what it does to the tail and to the load. A retry at p99
  costs one percent more traffic and can cut the tail in half, which is the trade hedging makes.
- Compute the percentiles over sliding windows rather than over the whole set, and watch what
  happens when a single backend degrades. Averaging percentiles across windows is wrong, and this
  is the fastest way to see why.